## 05_virtual_screening.ipynb

### What is virtual screening and when is it used

Virtual screening computationally filters millions of compounds to find
those most likely active before wet-lab testing. In industry, companies
screen 10M+ compounds in hours, choosing hundreds for synthesis.

Two types:
1. Structure-based: needs protein 3D structure (docking) - not used here
2. Ligand-based: uses known actives to train model (what we do)

This saves years and billions compared to testing everything experimentally.

### Load a screening library

from src.screener import load_screening_library

# Load from ZINC15 or SMILES file
library = load_screening_library(use_zinc=True, n_zinc_compounds=1000)
print(f"Loaded {len(library)} compounds from ZINC15")

### Apply drug-likeness filters

from src.screener import filter_by_drug_likeness

filtered, removed = filter_by_drug_likeness(library, rules="all")
print(f"Passed Lipinski+Veber+Ghose+Leadlike: {len(filtered)}")
print(f"Removed: {len(removed)}")

### Run the screening

from src.predictor import load_ensemble
model = load_ensemble('egfr').get('ensemble')

results_df = screen_library(
    model, filtered, target_name='egfr',
    featurization_method='combined',
    batch_size=256, top_n=100
)

print(results_df.head())

# Time the screening (placeholder)
import time
start = time.time()
# results = screen_library(...)
elapsed = time.time() - start
compounds_per_sec = len(library) / elapsed if elapsed > 0 else 0
print(f"Throughput: {compounds_per_sec:.0f} compounds/sec")

### Diversity filtering of hits

from src.screener import diversity_filter

top_hits = results_df.head(100)['smiles'].tolist()
diverse_hits = diversity_filter(top_hits, similarity_threshold=0.6)
print(f"Diversity filter: {len(top_hits)} -> {len(diverse_hits)} compounds")

### Generate screening report

from src.screener import generate_screening_report
generate_screening_report(
    results_df, 'egfr', 'ensemble',
    output_path='notebooks/screening_report.html'
)

### Enrichment analysis

from src.screener import enrichment_analysis

# Use known EGFR actives and inactives from dataset
known_actives = ["CCO", "CC(=O)OC1=CC=CC=C1C(=O)O"]  # Placeholder
known_inactives = ["C1CCCCC1", "CCCCCCCCC"]  # Placeholder

metrics = enrichment_analysis(
    model, known_actives, known_inactives,
    featurization_method='combined'
)

print(f"ROC-AUC: {metrics['roc_auc']:.3f}")
print(f"Enrichment Factor (1%): {metrics['enrichment_factor_1_percent']:.1f}x")
print(f"Enrichment Factor (5%): {metrics['enrichment_factor_5_percent']:.1f}x")
print(f"Enrichment Factor (10%): {metrics['enrichment_factor_10_percent']:.1f}x")